# Stacking Ensemble: K-Fold Meta-Learning for Iris Classification

This notebook demonstrates stacking, a meta-algorithm that trains base learners across k-fold cross-validation, then trains a meta-learner on out-of-fold predictions. This approach provides more robust meta-features than blending.

## 1. Stacking Algorithm Overview

**Stacking Process:**
1. For each fold in k-fold CV:
   - Train base learners on k-1 folds
   - Predict on hold-out fold → meta-features
2. Concatenate meta-features from all folds
3. Train base learners on full training data
4. Generate meta-features on test set
5. Train meta-learner on complete meta-features
6. Final prediction: meta-learner on test meta-features

**vs. Blending:** Stacking uses all training data for meta-features, avoiding data waste and improving generalization.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_predict, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)
print("Libraries loaded.")

## 2. Load and Prepare Data

In [ ]:
# Load Iris
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Generate Stacking Meta-Features (K-Fold CV)

In [ ]:
# Initialize base learners
base_learners = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

# K-fold for meta-feature generation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Generate meta-features using k-fold
print("Generating meta-features via 5-fold cross-validation...")
meta_features_train = np.zeros((X_train.shape[0], len(base_learners) * 3))

for i, (name, learner) in enumerate(base_learners.items()):
    # Get out-of-fold predictions
    oof_pred = cross_val_predict(learner, X_train, y_train, cv=kfold, method='predict_proba')
    meta_features_train[:, i*3:(i+1)*3] = oof_pred
    print(f"  {name}: meta-features shape {oof_pred.shape}")

print(f"\nTotal meta-features train shape: {meta_features_train.shape}")

## 4. Train Base Learners on Full Training Data

In [ ]:
# Train base learners on full training set
print("Training base learners on full training data...")
for name, learner in base_learners.items():
    learner.fit(X_train, y_train)
    acc = learner.score(X_test, y_test)
    print(f"  {name:20s}: test accuracy = {acc:.4f}")

## 5. Generate Test Meta-Features

In [ ]:
# Generate meta-features for test set
meta_features_test = np.column_stack(
    [learner.predict_proba(X_test) for learner in base_learners.values()]
)

print(f"Test meta-features shape: {meta_features_test.shape}")

In [ ]:
# Train meta-learner
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(meta_features_train, y_train)

print("Meta-learner trained on stacking meta-features.")

## 7. Evaluate Stacking Ensemble

In [ ]:
# Predictions
y_pred_stack = meta_learner.predict(meta_features_test)
stack_acc = accuracy_score(y_test, y_pred_stack)

print("\n=== ACCURACY COMPARISON ===")
print(f"Stacking Ensemble:        {stack_acc:.4f}")
print()

for name, learner in base_learners.items():
    y_pred = learner.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:20s}: {acc:.4f}")

best_base = max([accuracy_score(y_test, learner.predict(X_test)) for learner in base_learners.values()])
print(f"\nImprovement: {stack_acc - best_base:.4f} ({(stack_acc - best_base)*100:.2f}%)")

## 8. Classification Report

In [ ]:
print("Stacking Ensemble Classification Report:\n")
print(classification_report(y_test, y_pred_stack, target_names=target_names))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_stack)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Stacking Ensemble Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 10. Key Insights

- **Stacking vs. Blending:** Stacking uses k-fold CV for better meta-feature diversity
- **No data waste:** All training data contributes to meta-features
- **Computational cost:** Higher than blending due to k-fold training
- **Generalization:** Often better than both blending and individual base learners